[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/Escher_class/iJO1366_Escher_flux_exercise.ipynb)

# iJO1366 FBA and Escher flux export

Constraint-based simulation of *E. coli* **iJO1366** with COBRApy. Fluxes are saved as **JSON** files for visualization in [Escher](https://escher.github.io/).

## Exercises

1. Load iJO1366, run FBA, export fluxes.
2. Compare three carbon sources (glucose, glycerol, acetate) on aerobic medium.
3. Decrease oxygen uptake from 20 to 0 in five steps (glucose).
4. Knock out one reaction and compare to wild type.
5. Load the JSON files in Escher and interpret the maps.

## Setup

In [ ]:
import sys
!apt-get -qq install -y swig libgmp-dev 2>/dev/null
!{sys.executable} -m pip install -q cobra pandas numpy swiglpk

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from cobra.io import load_json_model

for folder in ['data', 'outputs/fluxes']:
    Path(folder).mkdir(parents=True, exist_ok=True)

MODEL_PATH = 'data/iJO1366.json'
MAP_PATH = 'data/iJO1366.Central metabolism.json'

!wget -q -O {MODEL_PATH} http://bigg.ucsd.edu/static/models/iJO1366.json
!wget -q -O "{MAP_PATH}" "http://bigg.ucsd.edu/escher_map_json/iJO1366.Central%20metabolism"

## Load model

Uptake is set with `reaction.lower_bound` (negative = uptake). Use `with model:` so each simulation reverts changes.

In [ ]:
model = load_json_model(MODEL_PATH)

print(model)
print(f'Reactions:   {len(model.reactions)}')
print(f'Metabolites: {len(model.metabolites)}')
print(f'Objective:   {model.objective}')

CARBON = {
    'glucose': 'EX_glc__D_e',
    'glycerol': 'EX_glyc_e',
    'acetate': 'EX_ac_e',
}
CARBON_UPTAKE = 10   # mmol/gDW/h
O2_AEROBIC = 20      # mmol/gDW/h
KNOCKOUT = 'ATPS4rpp'  # ATP synthase; try PFK or CS as well

def set_uptake(model, carbon_exchange, carbon_uptake=CARBON_UPTAKE, oxygen_uptake=O2_AEROBIC):
    """Close defined carbon exchanges; open one carbon source and O2 uptake."""
    for ex_id in CARBON.values():
        if model.reactions.has_id(ex_id):
            model.reactions.get_by_id(ex_id).lower_bound = 0
    if model.reactions.has_id(carbon_exchange):
        model.reactions.get_by_id(carbon_exchange).lower_bound = -carbon_uptake
    if model.reactions.has_id('EX_o2_e'):
        model.reactions.get_by_id('EX_o2_e').lower_bound = (
            -oxygen_uptake if oxygen_uptake > 0 else 0
        )
def save_flux_json(fluxes, path):
    """Escher single-dataset format: {reaction_id: flux, ...}"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(fluxes.to_dict(), f)

def save_flux_comparison(fluxes_a, fluxes_b, path):
    """Escher two-dataset format: [{...}, {...}]"""
    with open(path, 'w') as f:
        json.dump([fluxes_a.to_dict(), fluxes_b.to_dict()], f)

## 1. Carbon sources (aerobic)

Simulate growth on glucose, glycerol, and acetate. Save one JSON per carbon source and a comparison file (glucose vs glycerol).

In [ ]:
carbon_results = {}
summary = []

for name, exchange in CARBON.items():
    with model:
        set_uptake(model, exchange, oxygen_uptake=O2_AEROBIC)
        sol = model.optimize()
        carbon_results[name] = sol.fluxes
        save_flux_json(sol.fluxes, f'outputs/fluxes/flux_{name}_aerobic.json')
        summary.append({
            'condition': f'{name}_aerobic',
            'status': sol.status,
            'growth_rate': sol.objective_value,
            'carbon_source': exchange,
            'oxygen_uptake': O2_AEROBIC,
            'knockout': None,
        })

save_flux_comparison(
    carbon_results['glucose'],
    carbon_results['glycerol'],
    'outputs/fluxes/compare_glucose_vs_glycerol_aerobic.json',
)
save_flux_comparison(
    carbon_results['glucose'],
    carbon_results['acetate'],
    'outputs/fluxes/compare_glucose_vs_acetate_aerobic.json',
)

pd.DataFrame(summary)

## 2. Oxygen gradient (glucose)

Five aerobic-to-anaerobic steps: O₂ uptake 20 → 15 → 10 → 5 → 0 mmol/gDW/h.

In [ ]:
o2_steps = np.linspace(O2_AEROBIC, 0, 5)
o2_results = {}

for o2 in o2_steps:
    label = f'o2_{int(o2):02d}'
    with model:
        set_uptake(model, CARBON['glucose'], oxygen_uptake=float(o2))
        sol = model.optimize()
        o2_results[label] = sol.fluxes
        save_flux_json(sol.fluxes, f'outputs/fluxes/flux_glucose_{label}.json')
        summary.append({
            'condition': f'glucose_{label}',
            'status': sol.status,
            'growth_rate': sol.objective_value,
            'carbon_source': CARBON['glucose'],
            'oxygen_uptake': float(o2),
            'knockout': None,
        })

save_flux_comparison(
    o2_results['o2_20'],
    o2_results['o2_00'],
    'outputs/fluxes/compare_glucose_o2_20_vs_o2_00.json',
)

pd.DataFrame(summary)

## 3. Reaction knockout (glucose aerobic)

Knock out **ATPS4rpp** with `reaction.knock_out()` and compare to wild-type glucose aerobic.

In [ ]:
wt_fluxes = carbon_results['glucose']

if model.reactions.has_id(KNOCKOUT):
    with model:
        set_uptake(model, CARBON['glucose'], oxygen_uptake=O2_AEROBIC)
        model.reactions.get_by_id(KNOCKOUT).knock_out()
        sol = model.optimize()
        save_flux_json(sol.fluxes, f'outputs/fluxes/flux_mutant_{KNOCKOUT}_glucose_aerobic.json')
        save_flux_comparison(
            wt_fluxes,
            sol.fluxes,
            f'outputs/fluxes/compare_wt_vs_mutant_{KNOCKOUT}.json',
        )
        summary.append({
            'condition': f'mutant_{KNOCKOUT}_glucose_aerobic',
            'status': sol.status,
            'growth_rate': sol.objective_value,
            'carbon_source': CARBON['glucose'],
            'oxygen_uptake': O2_AEROBIC,
            'knockout': KNOCKOUT,
        })
else:
    print(f'Reaction {KNOCKOUT} not in model — try another ID.')

growth_summary = pd.DataFrame(summary)
growth_summary.to_csv('outputs/growth_summary.csv', index=False)
growth_summary

## Download outputs

In [ ]:
ZIP_NAME = 'iJO1366_escher_exercise_outputs.zip'
if Path(ZIP_NAME).exists():
  Path(ZIP_NAME).unlink()

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
  for folder in ['outputs', 'data']:
    for path in Path(folder).rglob('*'):
      if path.is_file():
        zf.write(path, arcname=str(path))

---

## Visualize in Escher

1. Open [https://escher.github.io/](https://escher.github.io/)
2. **Model → Load COBRA Model** → `data/iJO1366.json`
3. **Map → Load Map JSON** → `data/iJO1366.Central metabolism.json`
4. **Data → Load reaction data** → pick a JSON from `outputs/fluxes/`

| File | Content |
|------|---------|
| `flux_glucose_aerobic.json` | single flux map |
| `flux_glycerol_aerobic.json` | single flux map |
| `flux_acetate_aerobic.json` | single flux map |
| `compare_glucose_vs_glycerol_aerobic.json` | two datasets (side-by-side) |
| `flux_glucose_o2_XX.json` | O₂ gradient steps |
| `compare_glucose_o2_20_vs_o2_00.json` | aerobic vs anaerobic |
| `compare_wt_vs_mutant_ATPS4rpp.json` | wild type vs knockout |

In **Settings → Comparison**, choose *Difference* or *Log2(Fold Change)* when loading a two-dataset JSON.